# 0. 미션 개요

Hugging Face transformers 라이브러리를 사용하여 문서 요약 모델을 구현하는 미션입니다. 

데이터 로드 및 전처리부터 요약 모델 실행, 결과 평가까지 전체 파이프라인을 구축해 보세요.


# 1. 라이브러리 및 기본 설정

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
!pip install -q -U transformers datasets evaluate sentencepiece rouge_score accelerate bert-score

In [ ]:
import os  
import re 
import json 
import random 
import numpy as np  
import pandas as pd 
import evaluate  
import torch 
from datasets import Dataset, DatasetDict
from transformers import PreTrainedTokenizerFast, BartForConditionalGeneration, GenerationConfig, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback  

In [ ]:
# 시드 고정
SEED = 42  
random.seed(SEED)  
np.random.seed(SEED) 

# 2. 데이터 로드 및 전처리 함수 정의

## - 전처리

In [13]:
def clean_text(text):
    text = str(text)  # 혹시 모를 비문자 입력을 문자열로 통일
    text = text.replace("\n", " ").replace("\t", " ")  # 줄바꿈과 탭을 공백
    text = re.sub(r"[ ]{2,}", " ", text)  # 연속 공백을 하나로
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)  # 문장부호 앞의 불필요한 공백 제거
    text = re.sub(r"[『』「」《》]", " ", text)  # 특수 괄호류를 공백
    text = re.sub(r"[""'']", '"', text)            #  따옴표 통일
    text = re.sub(r"\d+,\d+", lambda m: m.group().replace(",",""), text)  # 1,000 → 1000
    text = re.sub(r"\s+", " ", text).strip()  # 마지막으로 전체 공백을 정리
    return text  

def load_json_dataset(file_path):
    with open(file_path, "r", encoding="utf-8") as f:  
        data = json.load(f) 

    examples = [] 

    for doc in data["documents"]: 
        sentences = []  

        for sublist in doc["text"]: 
            for item in sublist:  
                sentence = item.get("sentence", "")  
                if sentence:  
                    sentences.append(sentence)

        full_text = clean_text(" ".join(sentences))  
        summary = clean_text(doc["abstractive"][0] if doc["abstractive"] else "")  

        if len(full_text) >= 50 and len(summary) >= 10: 
            examples.append({
                "text": full_text,  
                "summary": summary  
            })

    return examples  


base_path = "/content/drive/MyDrive/data/summarization/"

# editorial(사설)
train_file = "train_original_editorial.json"
valid_file = "valid_original_editorial.json"

train_examples = load_json_dataset(os.path.join(base_path, train_file)) 
valid_examples = load_json_dataset(os.path.join(base_path, valid_file))  

print(f"학습 샘플 수: {len(train_examples)}") 
print(f"검증 샘플 수: {len(valid_examples)}")  

학습 샘플 수: 56756
검증 샘플 수: 7007


## - 데이터 길이 통계 확인

In [14]:
train_df = pd.DataFrame(train_examples)  
valid_df = pd.DataFrame(valid_examples) 

for df in [train_df, valid_df]:  
    df["text_len"] = df["text"].apply(len)  
    df["summary_len"] = df["summary"].apply(len)  

print("[Train 길이 통계]")
display(train_df[["text_len", "summary_len"]].describe())  

print("[Validation 길이 통계]")
display(valid_df[["text_len", "summary_len"]].describe())  


[Train 길이 통계]


,text_len,summary_len
count,56756.000000,56756.000000
mean,1170.513778,122.365072
std,241.090653,33.219312
min,109.000000,19.000000
25%,1017.000000,99.000000
50%,1134.000000,120.000000
75%,1289.000000,143.000000
max,1939.000000,397.000000


[Validation 길이 통계]


,text_len,summary_len
count,7007.000000,7007.000000
mean,1087.201798,126.159412
std,162.304951,32.363984
min,222.000000,33.000000
25%,1007.000000,104.000000
50%,1090.000000,125.000000
75%,1165.000000,146.500000
max,1859.000000,333.000000


## - Hugging Face Dataset 변환

In [15]:
train_dataset = Dataset.from_list(train_examples)  
valid_dataset = Dataset.from_list(valid_examples)  

dataset = DatasetDict({
    "train": train_dataset, 
    "validation": valid_dataset 
})

dataset  

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 56756
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 7007
    })
})

## - 모델 및 토크나이저 불러오기


In [16]:
checkpoint = "EbanLee/kobart-summary-v3"  

tokenizer = PreTrainedTokenizerFast.from_pretrained(checkpoint) 
model = BartForConditionalGeneration.from_pretrained(checkpoint)  

model.config.pad_token_id = tokenizer.pad_token_id  
model.config.eos_token_id = tokenizer.eos_token_id  


fallback_bos_token_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else (
    tokenizer.cls_token_id if tokenizer.cls_token_id is not None else tokenizer.eos_token_id
)  

model.config.bos_token_id = fallback_bos_token_id 
model.config.decoder_start_token_id = fallback_bos_token_id 

model.generation_config = GenerationConfig(
    max_length=128, 
    min_length=35,  
    num_beams=5, 
    length_penalty=1.1,  
    no_repeat_ngram_size=3,  
    early_stopping=True, 
    pad_token_id=tokenizer.pad_token_id, 
    eos_token_id=tokenizer.eos_token_id,  
    bos_token_id=fallback_bos_token_id, 
    decoder_start_token_id=fallback_bos_token_id  
)  

print("pad_token_id:", model.config.pad_token_id)  
print("eos_token_id:", model.config.eos_token_id)  
print("bos_token_id:", model.config.bos_token_id) 
print("decoder_start_token_id:", model.config.decoder_start_token_id) 


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

pad_token_id: 3
eos_token_id: 1
bos_token_id: 0
decoder_start_token_id: 0


## - 토큰화 및 모델 입력 변환


In [17]:
max_input_length = 576  
max_target_length = 140  

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["text"],  
        max_length=max_input_length,  
        truncation=True  
    )

    labels = tokenizer(
        text_target=examples["summary"],  
        max_length=max_target_length,  
        truncation=True 
    )

    model_inputs["labels"] = labels["input_ids"]  
    return model_inputs  

tokenized_datasets = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=dataset["train"].column_names,  
    desc="토큰화 진행 중"  
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,  
    model=model, 
    pad_to_multiple_of=8  
)

tokenized_datasets 


토큰화 진행 중:   0%|          | 0/56756 [00:00<?, ? examples/s]

토큰화 진행 중:   0%|          | 0/7007 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 56756
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7007
    })
})

# 3. 평가 함수 정의



In [18]:
rouge = evaluate.load("rouge") 

def compute_metrics(eval_pred):
    predictions, labels = eval_pred  

    if isinstance(predictions, tuple):  
        predictions = predictions[0]  

    if predictions.ndim == 3: 
        predictions = np.argmax(predictions, axis=-1)  

    predictions = np.nan_to_num(predictions, nan=0.0, posinf=0.0, neginf=0.0)  
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1).astype(np.int32)  
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)  

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)  
    labels = np.nan_to_num(labels, nan=tokenizer.pad_token_id, posinf=tokenizer.pad_token_id, neginf=tokenizer.pad_token_id) 
    labels = np.clip(labels, 0, tokenizer.vocab_size - 1).astype(np.int32) 
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)  

    decoded_preds = [pred.strip() for pred in decoded_preds] 
    decoded_labels = [label.strip() for label in decoded_labels]  

    result = rouge.compute(
        predictions=decoded_preds,  
        references=decoded_labels,  
        use_stemmer=False  
    )

    result = {key: round(float(value), 4) for key, value in result.items()} 
    return result 

# 4. 파인튜닝 설정 및 학습


In [19]:

output_dir = f"{base_path}/kobart_summary_outputs_final"  

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir, 
    num_train_epochs=5, 
    learning_rate=2e-5,  
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,  
    gradient_accumulation_steps=2, 
    weight_decay=0.05,  
    warmup_ratio=0.15,  
    lr_scheduler_type="cosine",  
    logging_steps=100,  
    eval_strategy="epoch",  
    save_strategy="epoch",  
    predict_with_generate=True,  
    generation_max_length=max_target_length,  
    generation_num_beams=5,  
    load_best_model_at_end=True, 
    metric_for_best_model="rougeL",  
    greater_is_better=True, 
    save_total_limit=2,  
    fp16=torch.cuda.is_available(),  
    report_to="none", 
    disable_tqdm=True,  
    label_smoothing_factor=0.1, 
    dataloader_num_workers=4  
)

trainer = Seq2SeqTrainer(
    model=model,  
    args=training_args,  
    train_dataset=tokenized_datasets["train"],  
    eval_dataset=tokenized_datasets["validation"],  
    data_collator=data_collator,  
    compute_metrics=compute_metrics, 
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [20]:
train_result = trainer.train() 
pd.DataFrame([train_result.metrics]) 


{'loss': '6.637', 'grad_norm': '5.386', 'learning_rate': '1.488e-06', 'epoch': '0.05637'}
{'loss': '6.2', 'grad_norm': '3.476', 'learning_rate': '2.99e-06', 'epoch': '0.1127'}
{'loss': '5.961', 'grad_norm': '3.455', 'learning_rate': '4.493e-06', 'epoch': '0.1691'}
{'loss': '5.897', 'grad_norm': '3.31', 'learning_rate': '5.995e-06', 'epoch': '0.2255'}
{'loss': '5.861', 'grad_norm': '3.476', 'learning_rate': '7.498e-06', 'epoch': '0.2818'}
{'loss': '5.796', 'grad_norm': '3.821', 'learning_rate': '9.001e-06', 'epoch': '0.3382'}
{'loss': '5.727', 'grad_norm': '3.459', 'learning_rate': '1.05e-05', 'epoch': '0.3946'}
{'loss': '5.716', 'grad_norm': '3.674', 'learning_rate': '1.201e-05', 'epoch': '0.451'}
{'loss': '5.719', 'grad_norm': '3.3', 'learning_rate': '1.351e-05', 'epoch': '0.5073'}
{'loss': '5.726', 'grad_norm': '3.452', 'learning_rate': '1.501e-05', 'epoch': '0.5637'}
{'loss': '5.676', 'grad_norm': '3.823', 'learning_rate': '1.651e-05', 'epoch': '0.6201'}
{'loss': '5.672', 'grad_norm

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '5.617', 'grad_norm': '3.294', 'learning_rate': '1.981e-05', 'epoch': '1.015'}
{'loss': '5.518', 'grad_norm': '3.633', 'learning_rate': '1.972e-05', 'epoch': '1.071'}
{'loss': '5.532', 'grad_norm': '3.555', 'learning_rate': '1.962e-05', 'epoch': '1.127'}
{'loss': '5.531', 'grad_norm': '3.283', 'learning_rate': '1.949e-05', 'epoch': '1.184'}
{'loss': '5.536', 'grad_norm': '3.581', 'learning_rate': '1.935e-05', 'epoch': '1.24'}
{'loss': '5.544', 'grad_norm': '3.313', 'learning_rate': '1.92e-05', 'epoch': '1.297'}
{'loss': '5.488', 'grad_norm': '3.333', 'learning_rate': '1.903e-05', 'epoch': '1.353'}
{'loss': '5.497', 'grad_norm': '3.542', 'learning_rate': '1.884e-05', 'epoch': '1.409'}
{'loss': '5.53', 'grad_norm': '3.114', 'learning_rate': '1.864e-05', 'epoch': '1.466'}
{'loss': '5.484', 'grad_norm': '3.406', 'learning_rate': '1.842e-05', 'epoch': '1.522'}
{'loss': '5.536', 'grad_norm': '3.182', 'learning_rate': '1.819e-05', 'epoch': '1.578'}
{'loss': '5.526', 'grad_norm': '3.1

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '5.407', 'grad_norm': '3.262', 'learning_rate': '1.586e-05', 'epoch': '2.029'}
{'loss': '5.335', 'grad_norm': '3.311', 'learning_rate': '1.551e-05', 'epoch': '2.086'}
{'loss': '5.351', 'grad_norm': '2.947', 'learning_rate': '1.516e-05', 'epoch': '2.142'}
{'loss': '5.369', 'grad_norm': '3.274', 'learning_rate': '1.48e-05', 'epoch': '2.198'}
{'loss': '5.398', 'grad_norm': '3.17', 'learning_rate': '1.443e-05', 'epoch': '2.255'}
{'loss': '5.334', 'grad_norm': '3.416', 'learning_rate': '1.405e-05', 'epoch': '2.311'}
{'loss': '5.327', 'grad_norm': '3.34', 'learning_rate': '1.367e-05', 'epoch': '2.368'}
{'loss': '5.304', 'grad_norm': '3.534', 'learning_rate': '1.328e-05', 'epoch': '2.424'}
{'loss': '5.341', 'grad_norm': '3.514', 'learning_rate': '1.288e-05', 'epoch': '2.48'}
{'loss': '5.36', 'grad_norm': '3.505', 'learning_rate': '1.248e-05', 'epoch': '2.537'}
{'loss': '5.314', 'grad_norm': '3.315', 'learning_rate': '1.207e-05', 'epoch': '2.593'}
{'loss': '5.346', 'grad_norm': '3.611

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '6471', 'train_samples_per_second': '43.85', 'train_steps_per_second': '1.371', 'train_loss': '5.561', 'epoch': '3'}


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


,train_runtime,train_samples_per_second,train_steps_per_second,train_loss,epoch
0,6471.0854,43.854,1.371,5.560643,3.0


# 5. 검증 데이터셋 평가


In [21]:
eval_results = trainer.evaluate()  
pd.DataFrame([eval_results]) 


{'eval_loss': '2.883', 'eval_rouge1': '0.276', 'eval_rouge2': '0.0989', 'eval_rougeL': '0.2727', 'eval_rougeLsum': '0.2724', 'eval_runtime': '1486', 'eval_samples_per_second': '4.715', 'eval_steps_per_second': '0.295', 'epoch': '3'}


,eval_loss,eval_rouge1,eval_rouge2,eval_rougeL,eval_rougeLsum,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,2.88305,0.276,0.0989,0.2727,0.2724,1486.1369,4.715,0.295,3.0


# 6. 정성 평가

In [22]:
model.eval()  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
model.to(device)  

for i in range(5):  
    sample_text = valid_examples[i]["text"] 
    gold_summary = valid_examples[i]["summary"]  

    inputs = tokenizer(
        sample_text, 
        return_tensors="pt",  
        max_length=max_input_length,  
        truncation=True  
    ).to(device)

    with torch.no_grad():  
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"], 
            num_beams=6,  
            max_length=max_target_length,
            min_length=20,  
            no_repeat_ngram_size=3,  
            length_penalty=1.2, 
            early_stopping=True 
        )

    pred_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)  

    print(f"[샘플 {i+1}]")
    print("원문:", sample_text[:500], "...") 
    print("정답 요약:", gold_summary)  
    print("생성 요약:", pred_summary)  


[샘플 1]
원문: 더불어민주당 이해찬 대표가 30 일 오후 국회에서 기자간담회를 열고 조국 전 법무부 장관 사태와 관련해 "국민 여러분께 매우 송구하다"고 밝혔다. 더불어민주당 이해찬 대표가 30 일 기자간담회를 열고 "조국 사태"와 관련, "국민 여러분께 매우 송구하다"는 입장을 밝혔다. 이 대표는 "검찰 개혁이란 대의에 집중하다 보니, 국민 특히 청년이 느꼈을 불공정에 대한 상대적 박탈감, 좌절감을 깊이 있게 헤아리지 못했다"며 "여당 대표로서 무거운 책임감을 느낀다"고 머리를 숙였다. 조국 전 법무부 장관이 14 일 사퇴한 이후 이 대표가 당 안팎의 쇄신 요구에 대해 입장을 표명한 것은 이번이 처음이다. 청와대와 여당은 "조국 정국"을 거치며 분출된 "공정"과 "정의"의 민심을 받들어 검찰 개혁에 매진하겠다면서도 두 달간 극심한 분열과 갈등을 초래한데 대해선 진지하게 성찰하는 모습을 보이지 않았다. 그나마 초선인 이철희 의원이 "당이 대통령 뒤에 비겁하게 숨어 있었다"고 비판했고, 표창원 의원은 ...
정답 요약: 이해찬 대표가 조국 사태와 관련 송구한 입장 표명이 과감한 인적 쇄신으로 이어져야 한다.
생성 요약: 더불어민주당 이해찬 대표가 30일 오후 국회에서 기자간담회를 열고 조국 전 법무부 장관 사태와 관련해 "국민 여러분께 매우 송구하다"는 입장을 밝혔으나 당 일각에선 "총선기획단장을 비롯한 당직 인선부터 쇄신 의지를 보여야 한다"는 비판의 목소리가 나오며 이날 유감 표명이 여권 전반의 대대적인 인적 쇄신으로 이어지길 기대한다.
[샘플 2]
원문: 탈원전 정책에서 비롯된 한국전력의 경영 부담이 결국 전기료 인상을 초래하게 됐다. 김종갑 한전 사장은 언론 인터뷰에서 "정부 정책에 따라 도입된 각종 전기료 특례할인을 모두 폐지하고 전기요금 원가공개 방안을 정부와 협의하겠다"고 밝혔다. 특례할인은 여름철 누진제, 주택용 절전 및 전기차 충전 등의 명목으로 전기료를 깎아주는 제도로서, 작년 1 조 1434 억원에 달했다. 이 제도가 없어지면 부담은 고스

# 7. 최종 정량 평가: ROUGE + BERTScore


In [24]:
bertscore = evaluate.load("bertscore") 

FINAL_EVAL_SIZE = min(200, len(valid_examples))  
eval_texts = [example["text"] for example in valid_examples[:FINAL_EVAL_SIZE]] 
eval_refs = [example["summary"] for example in valid_examples[:FINAL_EVAL_SIZE]]  
eval_preds = []  

for text in eval_texts: 
    inputs = tokenizer(
        text,
        return_tensors="pt",  
        max_length=max_input_length, 
        truncation=True  
    ).to(device)

    with torch.no_grad():  
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],  
            attention_mask=inputs["attention_mask"],  
            num_beams=5,  
            max_length=max_target_length,  
            min_length=35,  
            no_repeat_ngram_size=3,  
            length_penalty=1.1,  
            early_stopping=True  
        )

    pred_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)  
    eval_preds.append(pred_summary.strip())  

rouge_result = rouge.compute(
    predictions=eval_preds, 
    references=eval_refs, 
    use_stemmer=False  
)

bertscore_result = bertscore.compute(
    predictions=eval_preds,  
    references=eval_refs,  
    lang="ko"  
)

final_metrics = {
    "평가 샘플 수": FINAL_EVAL_SIZE, 
    "ROUGE-1": round(float(rouge_result["rouge1"]), 4),  
    "ROUGE-2": round(float(rouge_result["rouge2"]), 4),  
    "ROUGE-L": round(float(rouge_result["rougeL"]), 4),  
    "ROUGE-Lsum": round(float(rouge_result["rougeLsum"]), 4),  
    "BERTScore_Precision": round(float(np.mean(bertscore_result["precision"])), 4),  
    "BERTScore_Recall": round(float(np.mean(bertscore_result["recall"])), 4),  
    "BERTScore_F1": round(float(np.mean(bertscore_result["f1"])), 4)  
}

pd.DataFrame([final_metrics])  


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,평가 샘플 수,ROUGE-1,ROUGE-2,ROUGE-L,ROUGE-Lsum,BERTScore_Precision,BERTScore_Recall,BERTScore_F1
0,200,0.2748,0.0862,0.2732,0.2736,0.7634,0.7883,0.7752


# 최종 정리

이번 프로젝트에서는 Hugging Face Transformers를 활용하여 KoBART 기반 문서 요약 모델을 구축하고, 

데이터 로드부터 전처리, 토큰화, 모델 fine-tuning, 생성 결과 평가까지 전체 NLP 파이프라인을 구현하였다.  

최종 모델의 성능은 다음과 같다.

- ROUGE-1: 0.2748  
- ROUGE-2: 0.0862  
- ROUGE-L: 0.2732  
- ROUGE-Lsum: 0.2736  
- BERTScore F1: 0.7752  

평가 결과, ROUGE-1과 ROUGE-L이 약 0.27 수준으로 나타나 모델이 

원문의 핵심 키워드와 전반적인 문장 흐름을 비교적 안정적으로 반영하고 있음을 확인할 수 있었다. 

또한 BERTScore F1이 0.7752로 나타나, 생성된 요약문이 정답과 완전히 동일한 표현을 사용하지 않더라도 

의미적 유사성 측면에서는 충분히 신뢰할 수 있는 수준의 성능을 보였다.  

반면 ROUGE-2가 0.0862로 상대적으로 낮게 측정된 점을 통해, 

연속된 단어 표현이나 문장 단위의 정밀한 재현 능력은 제한적인 것으로 나타났다. 

이는 모델이 핵심 내용을 요약하는 데에는 효과적이지만, 참조 요약과 동일한 표현 구조를 생성하는 데에는 한계가 있음을 의미한다.  

종합적으로 본 모델은 의미 기반 요약 성능에서 강점을 보이며, 

실제 응용 환경에서도 핵심 정보 전달 중심의 요약에는 충분히 활용 가능한 수준으로 평가된다. 


추후 데이터 품질 개선, 도메인별 학습 전략, 생성 파라미터 최적화 등을 통해 표현 정밀도를 더욱 향상시킬 수 있을 것이다.  

이번 미션을 통해 문서 요약 모델의 전체 개발 과정과 평가 방법을 실습하며, Transformer 기반 생성 모델의 학습 및 성능 분석 능력을 체계적으로 확보할 수 있었다.